<a href="https://colab.research.google.com/github/anamacao/FAPESP-PIBIC-scrapping/blob/main/senadofederal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Coletando Notícias do Senado Federal

In [20]:
noticias_senado = []
url = SENADO_BASE_URL

print(f"📄 Coletando notícias do Senado: {url} (com rolagem infinita)")

driver.get(url)

# --- DEBUG: Print initial page source to inspect selectors ---
print("\n--- Debug: Initial Page Source (first 1000 chars) ---\n")
initial_soup = BeautifulSoup(driver.page_source, "html.parser")
print(initial_soup.prettify()[:1000])
print("\n----------------------------------------------------\n")
# --- END DEBUG ---

last_height = driver.execute_script("return document.body.scrollHeight")
scroll_attempts = 0
MAX_SCROLL_ATTEMPTS = 5 # Limit scrolling to prevent infinite loops

while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2) # Wait for page to load
    new_height = driver.execute_script("return document.body.scrollHeight")

    if new_height == last_height:
        scroll_attempts += 1
        if scroll_attempts > MAX_SCROLL_ATTEMPTS:
            print(f"❌ Atingido o limite de tentativas de rolagem sem novo conteúdo. Total de rolagens: {scroll_attempts}")
            break
    else:
        scroll_attempts = 0
    last_height = new_height

    # Extract current page source after scrolling
    soup = BeautifulSoup(driver.page_source, "html.parser")

    # Seletores para as notícias do Senado
    itens = soup.select("div.box-materia")

    print(f"   {len(itens)} notícias encontradas após rolagem")

    for item in itens:
        # título & link
        titulo_tag = item.select_one("h4 a")
        if not titulo_tag:
            continue

        titulo = titulo_tag.get_text(strip=True)
        link = titulo_tag["href"]

        # data e hora
        date_tag = item.select_one("div.data")
        data_raw = date_tag.get_text(strip=True) if date_tag else "NA"

        # Parse date from "Publicado em DD/MM/YYYY, às HHhMM"
        match = re.search(r'\d{2}/\d{2}/\d{4}, \w{2} (\d{2}h\d{2})', data_raw)
        if match:
            date_str_formatted = data_raw.replace('Publicado em ', '').replace('h', ':').replace(', às ', ' ').strip()
            try:
                parsed_date = datetime.strptime(date_str_formatted, "%d/%m/%Y %H:%M")
                formatted_date = parsed_date.strftime("%d/%m/%Y %H:%M")
            except ValueError:
                formatted_date = data_raw
        else:
            formatted_date = data_raw

        # extrai texto do artigo usando o seletor específico do Senado com Selenium
        paragrafos = extrair_paragrafos_com_selenium(link, "div#conteudo-materia p")

        # acumula
        # Check if news already exists to avoid duplicates from scrolling
        if not any(n['link'] == link for n in noticias_senado):
            noticias_senado.append({
                "titulo": titulo,
                "data": formatted_date,
                "link": link,
                "paragrafos": " || ".join(paragrafos),
                "fonte": SENADO_SOURCE
            })

            # grava no banco
            insert_article(
                title=titulo,
                date=formatted_date,
                author=SENADO_AUTHOR,
                url=link,
                source=SENADO_SOURCE
            )

    time.sleep(1)

print(f"\n✅ Total coletado do Senado: {len(noticias_senado)} notícias")

df_senado = pd.DataFrame(noticias_senado)
display(df_senado.head())

📄 Coletando notícias do Senado: https://www12.senado.leg.br/noticias/ultimas (com rolagem infinita)


MaxRetryError: HTTPConnectionPool(host='localhost', port=60303): Max retries exceeded with url: /session/d3b26985843ac73a45abeeb344bdd1e9/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=60303): Failed to establish a new connection: [Errno 111] Connection refused"))

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from datetime import datetime
import sqlite3

import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [2]:
SENADO_BASE_URL = "https://www12.senado.leg.br/noticias/ultimas"
SENADO_SOURCE = "Senado Federal"
SENADO_AUTHOR = "Agência Senado"

print("✅ Scraper do Senado Federal pronto!")

✅ Scraper do Senado Federal pronto!


In [3]:
def extrair_paragrafos_com_seletor(url, content_selector):
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        paragrafos = [p.get_text(strip=True) for p in soup.select(content_selector) if p.get_text(strip=True)]
        return paragrafos
    except requests.exceptions.RequestException as e:
        print(f"⚠️ Erro ao extrair parágrafos de {url}: {e}")
        return []

In [4]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

noticias_senado = []

url = "https://www12.senado.leg.br/noticias/ultimas"
print(f"📄 Coletando notícias do Senado: {url} (apenas as notícias visíveis na carga inicial da página)")

try:
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
except requests.exceptions.RequestException as e:
    print(f"⚠️ Erro de requisição ao acessar {url}: {e}.")
    soup = BeautifulSoup('', 'html.parser')
    r = type('obj', (object,), {'status_code' : 500})() # Mock a response object to avoid NameError later

if r.status_code == 200:
    soup = BeautifulSoup(r.text, "html.parser")
    # Seletores para as notícias do Senado
    itens = soup.select("div.box-materia")
    print(f"   {len(itens)} notícias encontradas")

    for item in itens:
        # título & link
        titulo_tag = item.select_one("h4 a")
        if not titulo_tag:
            continue

        titulo = titulo_tag.get_text(strip=True)
        link = titulo_tag["href"]

        # data e hora
        date_tag = item.select_one("div.data")
        data_raw = date_tag.get_text(strip=True) if date_tag else "NA"

        # Parse date from "Publicado em DD/MM/YYYY, às HHhMM"
        match = re.search(r'\d{2}/\d{2}/\d{4}, \w{2} (\d{2}h\d{2})', data_raw)
        if match:
            # Replacing 'h' with ':' to match datetime format
            date_str_formatted = data_raw.replace('Publicado em ', '').replace('h', ':').replace(', às ', ' ').strip()
            try:
                parsed_date = datetime.strptime(date_str_formatted, "%d/%m/%Y %H:%M")
                formatted_date = parsed_date.strftime("%d/%m/%Y %H:%M")
            except ValueError:
                formatted_date = data_raw # Fallback if parsing fails
        else:
            formatted_date = data_raw

        # extrai texto do artigo usando o seletor específico do Senado
        paragrafos = extrair_paragrafos_com_seletor(link, "div#conteudo-materia p")

        # acumula
        noticias_senado.append({
            "titulo": titulo,
            "data": formatted_date,
            "link": link,
            "paragrafos": " || ".join(paragrafos),
            "fonte": SENADO_SOURCE
        })

        # grava no banco
        insert_article(
            title=titulo,
            date=formatted_date,
            author=SENADO_AUTHOR,
            url=link,
            source=SENADO_SOURCE
        )

    time.sleep(1)

print(f"\n✅ Total coletado do Senado: {len(noticias_senado)} notícias")

df_senado = pd.DataFrame(noticias_senado)
display(df_senado.head())

📄 Coletando notícias do Senado: https://www12.senado.leg.br/noticias/ultimas (apenas as notícias visíveis na carga inicial da página)
   0 notícias encontradas

✅ Total coletado do Senado: 0 notícias


""


In [5]:
DATABASE_NAME = "internet_governance_news.db"

def create_database():
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS articles (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            date TEXT,
            author TEXT,
            url TEXT UNIQUE,
            source TEXT
        )
    """)
    conn.commit()
    conn.close()
    print("✅ Banco e tabela 'articles' prontos!")

create_database()

✅ Banco e tabela 'articles' prontos!


In [6]:
def insert_article(title, date, author, url, source):
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        cursor.execute("""
            INSERT INTO articles (title, date, author, url, source)
            VALUES (?, ?, ?, ?, ?)
        """, (title, date, author, url, source))
        conn.commit()
        print(f"✅ Artigo inserido: {title[:50]}...")
        return True
    except sqlite3.IntegrityError:
        print(f"⚠️ Artigo já existe (URL duplicada): {title[:50]}...")
        return False
    except Exception as e:
        print(f"❌ Erro ao inserir artigo '{title[:50]}...': {e}")
        return False
    finally:
        conn.close()

In [7]:
# %%
def load_articles_from_db():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("""
        SELECT *
        FROM articles
        ORDER BY date DESC
    """, conn)
    conn.close()
    return df

df_db = load_articles_from_db()
display(df_db.head())
print(f"📦 Total no banco: {len(df_db)} registros")

,id,title,date,author,url,source


📦 Total no banco: 0 registros


In [8]:
# %%
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

BASE_URL = "https://www.camara.leg.br/noticias/noticias-institucionais"
print("✅ Scraper da Câmara (Institucionais) pronto!")

✅ Scraper da Câmara (Institucionais) pronto!


In [9]:
# Carregar todas as notícias do banco de dados e filtrar pelo Senado
df_db_updated = load_articles_from_db()
df_senado_filtered = df_db_updated[df_db_updated['source'] == SENADO_SOURCE]
print(f"📦 Total no banco (apenas Senado): {len(df_senado_filtered)} registros")
display(df_senado_filtered.head(10))

📦 Total no banco (apenas Senado): 0 registros


,id,title,date,author,url,source


### Configuração do Selenium para Coleta do Senado

In [13]:
# Instalar Selenium e webdriver_manager
!pip install selenium webdriver_manager

# Instalar o Google Chrome e ChromeDriver necessários para o Colab
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" | tee /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install google-chrome-stable

OK
deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Get:8 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,214 B]
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 6,956 B in 1s (4,965 B/s)
Reading package lists... Done
W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg keyring (/

In [14]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager # Import webdriver_manager

# Setup Selenium Chrome options
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless') # Run in headless mode
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.add_argument('--verbose') # Add verbose logging for debugging
chrome_options.add_argument('--log-path=chromedriver.log') # Log to a file
chrome_options.add_argument('--disable-extensions') # Disable extensions
chrome_options.add_argument('--dns-prefetch-disable') # Disable DNS prefetch
chrome_options.add_argument('--disable-setuid-sandbox') # Disable setuid sandbox
chrome_options.add_argument('--remote-debugging-port=9222') # Specify a remote debugging port

# Explicitly set the path to the Chrome binary (now using google-chrome-stable)
chrome_options.binary_location = '/usr/bin/google-chrome'

# Configure Chrome service using webdriver_manager to get the correct chromedriver version
try:
    driver_path = ChromeDriverManager().install()
    print(f"ChromeDriver installed at: {driver_path}")
    webdriver_service = Service(executable_path=driver_path)
except Exception as e:
    print(f"⚠️ Erro ao instalar ChromeDriver: {e}")
    # Fallback or re-raise, depending on desired behavior
    raise

# Initialize the WebDriver
driver = webdriver.Chrome(service=webdriver_service, options=chrome_options)
print("✅ Selenium WebDriver configurado!")

ChromeDriver installed at: /root/.wdm/drivers/chromedriver/linux64/147.0.7727.56/chromedriver-linux64/chromedriver
✅ Selenium WebDriver configurado!


### Função de Extração de Parágrafos com Selenium

In [17]:
def extrair_paragrafos_com_selenium(url, content_selector):
    try:
        driver.get(url)
        # Wait for the content to be loaded, up to 10 seconds
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, content_selector))
        )
        soup = BeautifulSoup(driver.page_source, "html.parser")
        paragrafos = [p.get_text(strip=True) for p in soup.select(content_selector) if p.get_text(strip=True)]
        return paragrafos
    except Exception as e:
        print(f"⚠️ Erro ao extrair parágrafos de {url} com Selenium: {e}")
        return []


### Coleta de Notícias do Senado Federal com Rolagem Infinita

In [18]:
noticias_senado = []
url = SENADO_BASE_URL

print(f"📄 Coletando notícias do Senado: {url} (com rolagem infinita)")

driver.get(url)

last_height = driver.execute_script("return document.body.scrollHeight")
scroll_attempts = 0
MAX_SCROLL_ATTEMPTS = 5 # Limit scrolling to prevent infinite loops

while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2) # Wait for page to load
    new_height = driver.execute_script("return document.body.scrollHeight")

    if new_height == last_height:
        scroll_attempts += 1
        if scroll_attempts > MAX_SCROLL_ATTEMPTS:
            print(f"❌ Atingido o limite de tentativas de rolagem sem novo conteúdo. Total de rolagens: {scroll_attempts}")
            break
    else:
        scroll_attempts = 0
    last_height = new_height

    # Extract current page source after scrolling
    soup = BeautifulSoup(driver.page_source, "html.parser")

    # Seletores para as notícias do Senado
    itens = soup.select("div.box-materia")

    print(f"   {len(itens)} notícias encontradas após rolagem")

    for item in itens:
        # título & link
        titulo_tag = item.select_one("h4 a")
        if not titulo_tag:
            continue

        titulo = titulo_tag.get_text(strip=True)
        link = titulo_tag["href"]

        # data e hora
        date_tag = item.select_one("div.data")
        data_raw = date_tag.get_text(strip=True) if date_tag else "NA"

        # Parse date from "Publicado em DD/MM/YYYY, às HHhMM"
        match = re.search(r'\d{2}/\d{2}/\d{4}, \w{2} (\d{2}h\d{2})', data_raw)
        if match:
            date_str_formatted = data_raw.replace('Publicado em ', '').replace('h', ':').replace(', às ', ' ').strip()
            try:
                parsed_date = datetime.strptime(date_str_formatted, "%d/%m/%Y %H:%M")
                formatted_date = parsed_date.strftime("%d/%m/%Y %H:%M")
            except ValueError:
                formatted_date = data_raw
        else:
            formatted_date = data_raw

        # extrai texto do artigo usando o seletor específico do Senado com Selenium
        paragrafos = extrair_paragrafos_com_selenium(link, "div#conteudo-materia p")

        # acumula
        # Check if news already exists to avoid duplicates from scrolling
        if not any(n['link'] == link for n in noticias_senado):
            noticias_senado.append({
                "titulo": titulo,
                "data": formatted_date,
                "link": link,
                "paragrafos": " || ".join(paragrafos),
                "fonte": SENADO_SOURCE
            })

            # grava no banco
            insert_article(
                title=titulo,
                date=formatted_date,
                author=SENADO_AUTHOR,
                url=link,
                source=SENADO_SOURCE
            )

    time.sleep(1)

driver.quit() # Close the browser once scraping is done

print(f"\n✅ Total coletado do Senado: {len(noticias_senado)} notícias")

df_senado = pd.DataFrame(noticias_senado)
display(df_senado.head())

📄 Coletando notícias do Senado: https://www12.senado.leg.br/noticias/ultimas (com rolagem infinita)


MaxRetryError: HTTPConnectionPool(host='localhost', port=60303): Max retries exceeded with url: /session/d3b26985843ac73a45abeeb344bdd1e9/url (Caused by NewConnectionError("HTTPConnection(host='localhost', port=60303): Failed to establish a new connection: [Errno 111] Connection refused"))

In [22]:
print('--- Conteúdo do chromedriver.log ---')
try:
    with open('chromedriver.log', 'r') as f:
        print(f.read())
except FileNotFoundError:
    print('Arquivo chromedriver.log não encontrado. Verifique se o diretório de trabalho está correto ou se o arquivo foi gerado.')
print('------------------------------------')

--- Conteúdo do chromedriver.log ---
Arquivo chromedriver.log não encontrado. Verifique se o diretório de trabalho está correto ou se o arquivo foi gerado.
------------------------------------


In [23]:
print('--- Conteúdo do chromedriver.log ---')
try:
    with open('chromedriver.log', 'r') as f:
        print(f.read())
except FileNotFoundError:
    print('Arquivo chromedriver.log não encontrado. Verifique se o diretório de trabalho está correto ou se o arquivo foi gerado.')
print('------------------------------------')

--- Conteúdo do chromedriver.log ---
Arquivo chromedriver.log não encontrado. Verifique se o diretório de trabalho está correto ou se o arquivo foi gerado.
------------------------------------


In [21]:
# Close the browser once scraping is done
driver.quit()

In [ ]:
# Carregar todas as notícias do banco de dados e filtrar pelo Senado
df_db_updated = load_articles_from_db()
df_senado_filtered = df_db_updated[df_db_updated['source'] == SENADO_SOURCE]
print(f"📦 Total no banco (apenas Senado): {len(df_senado_filtered)} registros")
display(df_senado_filtered.head(10))